# Module 2: Customer Analytics & Predictive Modeling
Predicting customer annual spending from demographic and purchasing attributes.

## 1. Module 2 Introduction
This notebook presents an end-to-end customer analytics and predictive modeling workflow for the Zepto Data & AI Platform capstone project.

## 2. Business / Analytical Question
> **Central Analytical Question**: *Can we predict a customer's annual spending from their demographic and purchasing characteristics?*

The analysis focuses on understanding which customer characteristics are associated with annual spending and evaluating how accurately spending can be predicted.

## 3. Dataset Description
The raw dataset contains 1,020 rows representing 1,000 unique customer profiles, with 20 duplicate rows included to demonstrate data cleaning.

Features include age, income, membership duration, purchase count, average order value, discount usage, website visits, and preferred product category.

## 4. Import Libraries & Project-Relative Path Setup
Path handling uses `pathlib.Path` to resolve the project root independently of the current Jupyter execution directory.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Determine project root dynamically relative to notebook location
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent.parent
elif notebook_dir.name == "analytics":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analytics.src.data_loader import generate_synthetic_data, load_dataset, profile_dataset
from analytics.src.preprocessing import clean_dataset, engineer_features, prepare_data
from analytics.src.analysis import run_full_eda
from analytics.src.modeling import train_all_models
from analytics.src.evaluation import evaluate_and_export

## 5. Load Dataset

In [ ]:
data_path = project_root / "analytics" / "data" / "customer_spending.csv"
df_raw = load_dataset(data_path)
df_raw.head(5)

## 6. Dataset Profiling

In [ ]:
profile = profile_dataset(df_raw)
print(f"Shape: {profile['n_rows']} rows x {profile['n_cols']} cols")
print(f"Null values: {profile['null_counts']}")
print(f"Duplicate rows: {profile['num_duplicates']}")
df_raw.describe()

## 7. Data Cleaning
Removing duplicates, fixing invalid entries, and imputing missing values with medians.

In [ ]:
df_clean = clean_dataset(df_raw)
print(f"Cleaned dataset size: {len(df_clean)} rows")

## 8. Exploratory Data Analysis
Visualizing target distribution, feature correlation, and relationships.

In [ ]:
figures_dir = project_root / "analytics" / "output" / "figures"
fig_paths = run_full_eda(df_clean, output_dir=figures_dir)
for name, path in fig_paths.items():
    print(f"Generated plot: {name} -> {path}")

## 9. Feature Engineering
Creating non-leaking domain features: `purchase_frequency` and `visit_conversion_rate`.

*Note: The small constants (`+ 0.1` and `+ 1.0`) help avoid division-by-zero errors when the denominator is zero.*

In [ ]:
df_fe = engineer_features(df_clean)
df_fe.head(5)

## 10. Feature / Target Separation

In [ ]:
X = df_fe.drop(columns=["annual_spending", "customer_id"])
y = df_fe["annual_spending"]
print(f"Features shape: {X.shape}, Target shape: {y.shape}")

## 11. Train / Test Split (80/20)

In [ ]:
data_dict = prepare_data(df_fe, test_size=0.2, seed=42)
print(f"X_train shape: {data_dict['X_train_prep'].shape}, X_test shape: {data_dict['X_test_prep'].shape}")

## 12. Preprocessing Pipeline
Using ColumnTransformer with StandardScaler for numerical features and OneHotEncoder for categorical features.

In [ ]:
print("Feature names after one-hot encoding:")
print(data_dict["feature_names"])

## 13. Linear Regression (Baseline Model)

In [ ]:
models_out = train_all_models(data_dict)
print("Linear Regression baseline model trained.")

## 14. Random Forest Regression

In [ ]:
print("Random Forest Regressor trained (n_estimators=100).")

## 15. Model Evaluation Metrics (MAE, MSE, RMSE, R2)

In [ ]:
out_dir = project_root / "analytics" / "output"
eval_results = evaluate_and_export(models_out, data_dict, output_dir=out_dir)
eval_results

## 16. Model Comparison Table

In [ ]:
comparison_df = pd.DataFrame(eval_results).T
comparison_df

## 17. Feature Interpretation
Random Forest feature importances identify the key drivers of annual spending.

In [ ]:
rf_model = models_out["random_forest"]["model"]
importances = pd.DataFrame({"feature": data_dict["feature_names"], "importance": rf_model.feature_importances_})
importances.sort_values(by="importance", ascending=False).head(5)

## 18. Limitations
- Data is synthetically generated with normal noise assumptions.
- Real customer spending distributions often exhibit heavy tails and temporal seasonality.

## 19. Conclusion
The Random Forest Regressor achieved an R² of 0.9677 on the held-out test set. In this synthetic dataset, average order value, number of purchases, and income were among the most important predictive features. Results on real customer data may differ.